# Refocusing shuttle noise

Idea: since shuttling noise is all along one axis (Z axis), we can refocus it for the Z ancillae to turn it into an X error. This error will not propagate to the data qubits, so it only manifests as a measurement error on the ancilla (which can be mitigated by doing more measurement rounds). In this notebook, we test whether this works as expected.

In [1]:
import os
try:
    path_initialized
except NameError:
    path_initialized = True
    os.chdir('..')

import numpy as np
from sympy.abc import x, y

# from qldpc import codes
import networkx as nx
import matplotlib.pyplot as plt
import stim
import sinter

import src.device as device
import src.plotting as plotter
from src.RotatedSurfaceCode import RotatedSurfaceCode
from src.HGPCode import HGPCode
from src.QECCode import TestCode

In [ ]:
d = 5
hwp = device.hardware_params(1e-3)
dev = device.UnitCellDevice(d+2, d+2, hwp)
code = RotatedSurfaceCode(d)

# Want to optimize ancilla qubit schedules from the middle out, to ensure
# schedules mesh well together
cx,cy = tuple(np.mean(code.qubit_coords, axis=0))
anc_order = []
anc_queue = set(code.X_ancilla_indices)
while anc_queue:
    next_anc = min(anc_queue, key=lambda anc: np.linalg.norm([code.qubit_coords[anc][0]-cx, code.qubit_coords[anc][1]-cy]))
    anc_order.append(next_anc)
    anc_queue.remove(next_anc)
anc_queue = set(code.Z_ancilla_indices)
while anc_queue:
    next_anc = min(anc_queue, key=lambda anc: np.linalg.norm([code.qubit_coords[anc][0]-cx, code.qubit_coords[anc][1]-cy]))
    anc_order.append(next_anc)
    anc_queue.remove(next_anc)

data_coords = {q:((code.qubit_coords[q][0]+1)//2, (code.qubit_coords[q][1]+1)//2) for q in code.data_indices}
assert len(set(data_coords.values())) == len(data_coords)
sched = dev.compile_QEC_schedule(
    code,
    data_coords,
    code.check_cx_layers,
    rounds=1,
    use_highways=True,
    refocus_shuttle_noise=False,
    optimize_ancilla_start=False,
    separate_X_Z=True
)

[27, 29, 31, 32, 34, 36, 37, 39, 41, 42, 44, 46]
Optimizing 12 ancilla schedules............[25, 26, 28, 30, 33, 35, 38, 40, 43, 45, 47, 48]
Optimizing 12 ancilla schedules............